# Solving Graph Coloring Problems Using Grover's Algorithm Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Solving Graph Coloring Problems Using Grover's Algorithm" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [34]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import QPU, Qubits


## Problem 1. Is vertex coloring valid? (classical)

A graph coloring is valid when the vertices connected by every edge have a different color. This means that we have to check every edge, see if its vertices have the same color, and if it is the case, return that the graph coloring is invalid. If every edge passed the test, we can safely say the graph coloring is valid.

Since the color of vertex $n$ is the $n$-th element of the `colors` array, we simply loop through every edge, which is a pair of vertex indices, and compare the colors with those indices.

In [ ]:
def is_vertex_coloring_valid(V: int, edges: list[tuple[int, int]], colors: list[int]) -> bool:
    for (v0, v1) in edges:
        if colors[v0] == colors[v1]:
            return False
    return True

## Problem 2. Read coloring from a Qubits register

The solution to this exercise consists of two parts:

1. Split the given qubit register into chunks of length `n_bits`. 

2. Read an integer from a register of qubits of length `n_bits`. Since you are guaranteed that the qubits in the register are in a basis state, simply measuring them will give you the necessary information and leave the state of the qubits unchanged. You can use `res = reg.read()` to measure each qubit in the register without resetting it to the $\ket{0}$ state afterwards. This method measures each qubit in the register separately and returns an integer that corresponds to the bit string composed of individual measurement results in little-endian notation (the least significant bit stored first).

In [ ]:
def read_coloring(n_bits: int, qs: Qubits) -> list[int]:
    # Split register into segements of size n_bits
    V = qs.num_qubits // n_bits
    color_partitions = [qs[i * n_bits:(i + 1) * n_bits] for i in range(V)]

    # Read an integer from each segemnt
    return [qi.read() for qi in color_partitions]

## Problem 3. Are colors equal?

The goal is to flip the qubit $\ket{y}$ if and only if each of the matching pairs of qubits in the registers `x0` and `x1` are in the same state.

You can check whether two qubits are in the same state by computing their $\textrm{XOR}$: if their state was the same, their $\textrm{XOR}$ will be $0$. You can use the $\textrm{CNOT}$ gate to compute $\textrm{XOR}$ of two qubits in place, for example, using the qubit of the register `x0` as the control and the qubit of the register `x1` as the target.

Once you've used $n\_bits$ $\textrm{CNOT}$ gates to compute all pairwise $\textrm{XOR}$'s, you'll need to flip the target qubit $\ket{y}$ only if all  qubits in `x1` are in the $\ket{0}$ state. This can be done by using zero-controlled $X$ gate, that is, `y.x(cond = x1 == 0)`.

Finally, you need to uncompute the bitwise $\textrm{XOR}$'s to ensure that the qubits in `x1` are returned to their original state.

In [ ]:
def oracle_color_equality(x0 : Qubits, x1 : Qubits, y : Qubits) -> None:
    for i in range(len(x0)):
        x1[i].x(cond = x0[i])
    
    y.x(cond=x1 == 0)

    for i in range(len(x0)):
        x1[i].x(cond = x0[i])

## Problem 4. Is vertex coloring valid? (quantum)

You need to allocate a register of auxiliary qubits, one for each edge of the graph, that will have their states flipped if the vertices connected by the corresponding edge have the same color. This check can be done easily for each edge using with the `oracle_color_equality` from the previous task.

Then, you need to check if these auxiliary qubits are still in state $\ket{0 \cdots 0}$; if they are, all the necessary pairs of colors are distinct, the coloring is valid.

Since the coloring is provided as a Qubits register, with two qubits per vertex ($2$ qubits = $4$ basis states = $4$ colors), you have to take the correct chunks of the coloring to extract the color of each vertex. You can deduce that the coloring of vertex $j$ is encoded in qubits in positions $2j$ and $2j + 1$.

Make sure to uncompute the changes to the auxiliary qubits after you evaluate the final result to leave them clean before their release.

In [ ]:
def oracle_vertex_coloring(V: int, edges: list[(int,int)], x: Qubits, y: Qubits) -> None:
    edgesNumber = len(edges)
    if edgesNumber == 0:
        y.x()
        return

    conflicts = Qubits(edgesNumber, "conflicts", x.qpu)

    for i in range(edgesNumber):
        v0, v1 = edges[i]
        oracle_color_equality(x[2 * v0 : 2 * (v0 + 1)], x[2 * v1 : 2 * (v1 + 1)], conflicts[i])
    
    y.x(cond=conflicts == 0)

    # Uncompute
    for i in range(edgesNumber):
        v0, v1 = edges[i]
        oracle_color_equality(x[2 * v0 : 2 * (v0 + 1)], x[2 * v1 : 2 * (v1 + 1)], conflicts[i])

    conflicts.release()

## Problem 5. Is weak coloring valid? (classical)

A weak coloring is valid when each of the vertices is weakly colored, that is, has either no neighbors or has at least one neighbor of a different color. This means that the solution needs to iterate through all vertices and check this condition for each of them, tracking separately the number of vertices connected to it with an edge and the existence of a connected vertex of a different color.

In [ ]:
def is_weak_coloring_valid(V : int, edges: list[tuple[int, int]], colors: list[int]) -> bool:
    for vertex in range(V):
        neighbor_count = 0
        has_different_neighbor = False

        for start, end in edges:
            if start == vertex or end == vertex:
                neighbor_count += 1
                if colors[start] != colors[end]:
                    has_different_neighbor = True

        if neighbor_count > 0 and not has_different_neighbor:
            return False
    
    return True   

## Problem 6. Is one-vertex weak coloring valid? (quantum)

To implement this check, you need to start by counting the vertices connected to the given vertex and allocating a register of auxiliary qubits, one for each neighboring vertex. These qubits will have their states flipped if the vertices connected by the corresponding edge have the same color. This can be done easily using the `oracle_color_equality` from an earlier problem, similarly to how you did it when validating the vertex coloring of the graph.

Now, for the coloring of the vertex to be valid, it needs to either have no neighbors, or to have at least one neighbor of a different color, so the register of auxiliary qubits should be in any state but $\ket{1...1}$. To implement this, you can flip the target qubit $\ket{y}$ unconditionally using the $X$ gate, and then flip it again only if qubits are in state $\ket{1...1}$.

Make sure to uncompute the changes to the auxiliary qubits after you evaluate the final result to leave them clean before their release.

In [ ]:
def oracle_weak_coloring_one_vertex(V: int, edges: list[tuple[int, int]], x: Qubits, y: Qubits, vertex: int) -> None:
    neighbor_edges = []
    for (a, b) in edges:
        if a == vertex or b == vertex:
            neighbor_edges.append((a, b))
    
    n_neighbors = len(neighbor_edges)
    
    y.x()
    if n_neighbors > 0:
        same_color_checks = Qubits(n_neighbors, "same_color", x.qpu)

        for ((a, b), check_qubit) in zip(neighbor_edges, same_color_checks):
            oracle_color_equality(x[2 * a:2 * (a + 1)], x[2 * b:2 * (b + 1)], check_qubit)
        
        y.x(cond=same_color_checks)

        # Uncompute
        for ((a, b), check_qubit) in zip(neighbor_edges, same_color_checks):
            oracle_color_equality(x[2 * a:2 * (a + 1)], x[2 * b:2 * (b + 1)], check_qubit)

        same_color_checks.release()

## Problem 7. Is weak coloring valid? (quantum)

To implement this oracle, you need to check whether each vertex is colored correctly and combine the results of individual checks. You'll need to allocate a register of qubits, one per vertex, that will be flipped if the corresponding vertex is weakly colored. This check can be done easily using the `oracle_weak_coloring_one_vertex` operation defined in the previous task. Then, you need to check if all qubits in the register is in state $\ket{1...1}$ using the controlled $X$ gate; if they are, the coloring is valid.

As usual, remember to uncompute the changes to the auxiliary qubits after you evaluate the final result to leave them clean before their release.

In [ ]:
def oracle_weak_coloring(V: int, edges: list[tuple[int, int]], x: Qubits, y: Qubits) -> None:
    valid_vertices = Qubits(V, "valid_vertices", x.qpu)

    for v in range(V):
        oracle_weak_coloring_one_vertex(V, edges, x, valid_vertices[v], v)
 
    y.x(cond=valid_vertices)
    
    # Uncompute
    for v in range(V):
        oracle_weak_coloring_one_vertex(V, edges, x, valid_vertices[v], v)

    valid_vertices.release()

> Copyright (c) 2026 PsiQuantum